# Experiment 4cii: Concentrated Attention Mechanisms

Compare 5 attention concentration approaches against the baseline (4c) GAT:

1. **Temperature scaling** — `softmax(scores / T)` with T=0.25
2. **Entropy regularization** — penalty on attention entropy in the loss
3. **Sparsemax** — simplex projection producing exact zeros
4. **Top-k** — keep only top-k logits before softmax
5. **Power sharpening** — raise weights to power > 1 and renormalize

All use the same architecture as 4c (LSTM-GATv2 with rolling Pearson mask).

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from empyrical import sharpe_ratio, sortino_ratio, max_drawdown, annual_return, annual_volatility, calmar_ratio
import random, tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
SEED = 40
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023
VOL_TARGET = 0.15

TRAIN_STRIDE = 20
SCALE_SCORES = True

# Rolling Pearson Configuration
CORRELATION_LOOKBACK = 20
CORRELATION_THRESHOLD = 0.4
USE_EQUITY_RETURNS = True

# Model Configuration
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GATv2 Hyperparameters (same as 4c)
HIDDEN_LAYER_SIZE = 10
GAT_UNITS = 16
ATTN_HEADS = 4
LSTM_DROPOUT = 0.4
ATTN_DROPOUT = 0.1
LEARNING_RATE = 0.0008
MAX_GRADIENT_NORM = 1.0
BATCH_SIZE = 57

if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
else:
    RESULTS_BASE = "FINAL_RESULTS"

returns_type = "equity" if USE_EQUITY_RETURNS else "straddle"
CONFIG_NAME = f"lb{CORRELATION_LOOKBACK}_th{CORRELATION_THRESHOLD}_{returns_type}"

# --- Concentrated attention approaches ---
APPROACHES = {
    "baseline": {
        "attention_type": "softmax",
    },
    "temperature_0.25": {
        "attention_type": "softmax",
        "temperature": 0.25,
    },
    "entropy_reg_0.1": {
        "attention_type": "softmax",
        "lambda_entropy": 0.1,
    },
    "sparsemax": {
        "attention_type": "sparsemax",
    },
    "top_k_10": {
        "attention_type": "softmax",
        "top_k": 10,
    },
    "power_2.0": {
        "attention_type": "softmax",
        "sharpen_power": 2.0,
    },
}

print(f"Experiment: 4cii Concentrated Attention")
print(f"Config: {CONFIG_NAME}")
print(f"Approaches: {list(APPROACHES.keys())}")

## 3. Helper Functions

In [ ]:
def calc_daily_returns(df, returns_col="captured_returns"):
    num_tickers = df["identifier"].nunique()
    daily_ret = df.groupby("time")[returns_col].sum() / num_tickers
    daily_ret.index = pd.to_datetime(daily_ret.index)
    return daily_ret.sort_index()

def calc_vol_scaled_returns(daily_returns, target_vol=0.15):
    current_vol = daily_returns.std() * np.sqrt(252)
    return daily_returns * (target_vol / current_vol) if current_vol > 0 else daily_returns

def calc_metrics(daily_returns, name="Strategy"):
    return {"Strategy": name, "E[Ret.]": annual_return(daily_returns),
        "Vol.": annual_volatility(daily_returns), "Sharpe": sharpe_ratio(daily_returns),
        "Sortino": sortino_ratio(daily_returns), "Max DD": -max_drawdown(daily_returns),
        "Calmar": calmar_ratio(daily_returns), "Hit Rate": (daily_returns > 0).mean(),
        "Avg P/L": daily_returns[daily_returns > 0].mean() / abs(daily_returns[daily_returns < 0].mean()) if (daily_returns < 0).any() else np.nan}

def calc_metrics_vol_normalized(daily_returns, name, target_vol=0.15):
    scaled = calc_vol_scaled_returns(daily_returns, target_vol)
    return calc_metrics(scaled, name + " (Vol-Norm)"), scaled

def display_metrics(m):
    df = pd.DataFrame([m]).set_index("Strategy")
    for c in ["E[Ret.]","Vol.","Max DD","Hit Rate"]:
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.2%}")
    for c in ["Sharpe","Sortino","Calmar","Avg P/L"]:
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.3f}")
    display(df)

def calc_yearly_sharpes(daily_returns):
    return {y: sharpe_ratio(daily_returns[daily_returns.index.year == y]) for y in sorted(daily_returns.index.year.unique())}

def plot_results(daily_returns_dict, title):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = plt.cm.tab10(np.linspace(0, 1, len(daily_returns_dict)))
    for (n, r), c in zip(daily_returns_dict.items(), colors):
        axes[0,0].plot(((1+r).cumprod()-1).index, ((1+r).cumprod()-1).values, label=n, lw=1.5, color=c)
        cum = (1+r).cumprod(); dd = (cum - cum.cummax()) / cum.cummax()
        axes[0,1].fill_between(dd.index, dd.values, 0, alpha=0.3, label=n, color=c)
        rs = r.rolling(252).mean() / r.rolling(252).std() * np.sqrt(252)
        axes[1,0].plot(rs.index, rs.values, label=n, lw=1, color=c)
    axes[0,0].set_title("Cumulative Returns"); axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)
    axes[0,1].set_title("Drawdown"); axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)
    axes[1,0].axhline(y=0, color="black", ls="--", lw=0.5); axes[1,0].set_title("Rolling Sharpe"); axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)
    yd = pd.DataFrame({n: calc_yearly_sharpes(r) for n,r in daily_returns_dict.items()})
    yd.plot(kind="bar", ax=axes[1,1], width=0.8); axes[1,1].axhline(y=0, color="black", ls="--", lw=0.5)
    axes[1,1].set_title("Yearly Sharpe"); axes[1,1].tick_params(axis="x", rotation=45); axes[1,1].grid(True, alpha=0.3, axis="y")
    plt.suptitle(title, fontsize=14, fontweight="bold"); plt.tight_layout(); plt.show()

## 4. Data Loading (Rolling Pearson)

In [ ]:
features_path = "data/straddle_features/features.csv"
if 'google.colab' in str(get_ipython()):
    features_path = "/content/drive/MyDrive/features.csv"

df = pd.read_csv(features_path)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df)} rows, {df['ticker'].nunique()} tickers")

In [ ]:
from gml.graph_model_inputs import RollingGraphModelFeatures

rolling_features = RollingGraphModelFeatures(
    df=df,
    total_time_steps=TOTAL_TIME_STEPS,
    correlation_lookback=CORRELATION_LOOKBACK,
    correlation_threshold=CORRELATION_THRESHOLD,
    returns_column="daily_returns",
    use_equity_returns=USE_EQUITY_RETURNS,
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True,
    time_features=False,
)

datasets = rolling_features.make_rolling_graph_dataset(sliding_window=True)
train_data = datasets["train"]
valid_data = datasets["valid"]
test_data = datasets["test"]

num_tickers = train_data["inputs"].shape[1]
time_steps = train_data["inputs"].shape[2]
input_size = train_data["inputs"].shape[3]

print(f"Training: inputs={train_data['inputs'].shape}, adj={train_data['adjacency'].shape}")
print(f"Validation: inputs={valid_data['inputs'].shape}, adj={valid_data['adjacency'].shape}")
print(f"Test: inputs={test_data['inputs'].shape}, adj={test_data['adjacency'].shape}")
print(f"num_tickers={num_tickers}, time_steps={time_steps}, input_size={input_size}")

## 5. Training Loop

In [ ]:
from gml.graph_attention_v2 import build_lstm_gat_rolling_concentrated, extract_attention_weights_concentrated
from gml.experiment_utils import compute_graph_stats, save_experiment_results

X_train = [train_data["inputs"], train_data["adjacency"]]
y_train = train_data["outputs"]
w_train = train_data["active_entries"]

X_valid = [valid_data["inputs"], valid_data["adjacency"]]
y_valid = valid_data["outputs"]
w_valid = valid_data["active_entries"]

X_test = [test_data["inputs"], test_data["adjacency"]]
test_dates_arr = pd.to_datetime(test_data["date"][:, 0, -1, 0])

# Lightweight storage: only metrics + attention stats (not full models/weights)
all_results = {}

def run_approach(approach_name, approach_kwargs):
    """Train one approach, save to drive, store lightweight results, free memory."""
    print("=" * 70)
    print(f"APPROACH: {approach_name}")
    print(f"  kwargs: {approach_kwargs}")
    print("=" * 70)

    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

    model = build_lstm_gat_rolling_concentrated(
        num_tickers=num_tickers, time_steps=time_steps, input_size=input_size,
        hidden_layer_size=HIDDEN_LAYER_SIZE, gat_units=GAT_UNITS, attn_heads=ATTN_HEADS,
        lstm_dropout=LSTM_DROPOUT, attn_dropout=ATTN_DROPOUT,
        learning_rate=LEARNING_RATE, max_gradient_norm=MAX_GRADIENT_NORM,
        use_residual=False, scale_scores=SCALE_SCORES,
        **approach_kwargs,
    )
    print(f"  Parameters: {model.count_params():,}")

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True, verbose=1,
    )

    history = model.fit(
        X_train, y_train, sample_weight=w_train,
        validation_data=(X_valid, y_valid, w_valid),
        epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[early_stopping], verbose=1,
    )

    # Predict
    predictions = model.predict(X_test)

    # Build results df
    positions = predictions[:, :, -1, 0].reshape(-1)
    returns = test_data["outputs"][:, :, -1, 0].reshape(-1)
    captured_returns = positions * returns
    dates = test_data["date"][:, :, -1, 0].reshape(-1)
    identifiers = test_data["identifier"][:, :, -1, 0].reshape(-1)

    results_df = pd.DataFrame({
        "time": dates, "identifier": identifiers,
        "position": positions, "returns": returns, "captured_returns": captured_returns,
    })
    results_df["time"] = pd.to_datetime(results_df["time"])
    results_df = results_df[results_df["identifier"] != "0"]

    daily_returns = calc_daily_returns(results_df)
    metrics_raw = calc_metrics(daily_returns, approach_name)

    # Extract attention weights
    attn_type = approach_kwargs.get("attention_type", "softmax")
    attn_weights = extract_attention_weights_concentrated(
        model, X_test, gat_layer_name="concentrated_gat",
        attention_type=attn_type,
    )

    # --- Save to drive immediately ---
    exp_name = f"4cii_{approach_name}"
    hyperparams = {
        "hidden_layer_size": HIDDEN_LAYER_SIZE, "gat_units": GAT_UNITS,
        "attn_heads": ATTN_HEADS, "lstm_dropout": LSTM_DROPOUT,
        "attn_dropout": ATTN_DROPOUT, "learning_rate": LEARNING_RATE,
        "max_gradient_norm": MAX_GRADIENT_NORM, "batch_size": BATCH_SIZE,
        "residual": False,
        "model_type": "GATv2_rolling_concentrated",
        "correlation_lookback": CORRELATION_LOOKBACK,
        "correlation_threshold": CORRELATION_THRESHOLD,
        "use_equity_returns": USE_EQUITY_RETURNS,
        "total_time_steps": TOTAL_TIME_STEPS,
        "train_start": TRAIN_START, "test_start": TEST_START, "test_end": TEST_END,
        **approach_kwargs,
    }
    metrics_norm, _ = calc_metrics_vol_normalized(daily_returns, exp_name, VOL_TARGET)
    yearly_sharpes = calc_yearly_sharpes(daily_returns)
    avg_attn = attn_weights.mean(axis=1)
    graph_stats = compute_graph_stats(avg_attn, threshold=0.0)

    save_experiment_results(
        experiment_name=exp_name, seed=SEED, config_name=CONFIG_NAME,
        predictions=predictions, results_df=results_df,
        daily_returns=daily_returns, metrics_raw=metrics_raw,
        metrics_norm=metrics_norm, yearly_sharpes=yearly_sharpes,
        training_history=history.history, hyperparams=hyperparams,
        test_dates=test_dates_arr.values,
        attention_weights=attn_weights,
        adjacency=test_data["adjacency"],
        graph_stats=graph_stats,
        model=model,
        base_dir=RESULTS_BASE,
    )
    print(f"  Saved: {exp_name}")

    # --- Keep lightweight results for comparison plots ---
    all_results[approach_name] = {
        "daily_returns": daily_returns,
        "metrics": metrics_raw,
        "history_loss": history.history["loss"],
        "history_val_loss": history.history["val_loss"],
        "concentration": {
            "avg_attn_sample": avg_attn[:50].copy(),  # first 50 for heatmaps
            "graph_stats": graph_stats,
        },
        "kwargs": approach_kwargs,
    }

    # --- Free memory ---
    del model, predictions, attn_weights, avg_attn, results_df
    tf.keras.backend.clear_session()
    import gc; gc.collect()

    print(f"  Sharpe: {metrics_raw['Sharpe']:.4f}")
    print(f"  Memory freed.\n")

### 5a. Baseline (standard softmax)

In [ ]:
run_approach("baseline", APPROACHES["baseline"])

### 5b. Temperature Scaling (T=0.25)

In [ ]:
run_approach("temperature_0.25", APPROACHES["temperature_0.25"])

### 5c. Entropy Regularization (lambda=0.1)

In [ ]:
run_approach("entropy_reg_0.1", APPROACHES["entropy_reg_0.1"])

### 5d. Sparsemax

In [ ]:
run_approach("sparsemax", APPROACHES["sparsemax"])

### 5e. Top-k (k=10)

In [ ]:
run_approach("top_k_10", APPROACHES["top_k_10"])

### 5f. Power Sharpening (p=2.0)

In [ ]:
run_approach("power_2.0", APPROACHES["power_2.0"])

### 5g. Temperature Sweep

T=0.25 was the best-performing approach. Sweep T=[0.05, 0.1, 0.15, 0.25, 0.5, 0.75] to find the optimal temperature and map the concentration-performance tradeoff.

In [ ]:
TEMP_SWEEP = [0.05, 0.1, 0.15, 0.25, 0.5, 0.75]

for T in TEMP_SWEEP:
    name = f"temp_{T}"
    # Skip if already run (e.g. T=0.25 was done as "temperature_0.25")
    if name in all_results:
        print(f"Skipping {name} (already in all_results)")
        continue
    
    run_approach(name, {"attention_type": "softmax", "temperature": T})

In [ ]:
# Temperature sweep summary: Sharpe and concentration vs temperature
sweep_names = [f"temp_{T}" for T in TEMP_SWEEP]
# Also include the original temperature_0.25 if it exists but temp_0.25 doesn't
available = [n for n in sweep_names if n in all_results]
if "temperature_0.25" in all_results and "temp_0.25" not in all_results:
    all_results["temp_0.25"] = all_results["temperature_0.25"]
    available = [n for n in sweep_names if n in all_results]

# Compute within-mask concentration for sweep approaches
adj_sample = test_data["adjacency"][:50]
sweep_data = []
for name in available:
    r = all_results[name]
    avg_attn = r["concentration"]["avg_attn_sample"]
    if avg_attn is None:
        continue
    conc = compute_concentration_metrics(avg_attn, adj_sample[:len(avg_attn)])
    T = float(name.split("_")[1])
    sweep_data.append({
        "temperature": T,
        "sharpe": r["metrics"]["Sharpe"],
        "entropy_ratio": conc["entropy_ratio"],
        "within_mask_gini": conc["within_mask_gini"],
        "within_mask_eff_neighbors": conc["within_mask_eff_neighbors"],
        "within_mask_max_weight": conc["within_mask_max_weight"],
    })

sweep_df = pd.DataFrame(sweep_data).sort_values("temperature")
print("Temperature Sweep Results:")
print(sweep_df.to_string(index=False, float_format="%.4f"))

# Plot: Sharpe and entropy ratio vs temperature
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(sweep_df["temperature"], sweep_df["sharpe"], "o-", color="#d62728", lw=2, ms=8, label="Sharpe")
ax1.set_xlabel("Temperature", fontsize=12)
ax1.set_ylabel("Sharpe Ratio", color="#d62728", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#d62728")

ax2 = ax1.twinx()
ax2.plot(sweep_df["temperature"], sweep_df["entropy_ratio"], "s--", color="#1f77b4", lw=2, ms=8, label="Entropy Ratio")
ax2.set_ylabel("Entropy Ratio (1=uniform)", color="#1f77b4", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#1f77b4")

# Add baseline reference
if "baseline" in all_results:
    ax1.axhline(y=all_results["baseline"]["metrics"]["Sharpe"], color="#d62728", ls=":", alpha=0.5, label="Baseline Sharpe")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
ax1.set_title("Temperature Sweep: Performance vs Concentration", fontsize=13, fontweight="bold")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Performance Comparison

### Load saved results from Drive (after kernel restart)

Run this cell to reload all previously saved 4cii results without re-training.

In [ ]:
# Reload saved results from Drive (run after kernel restart instead of re-training)
from gml.experiment_utils import load_experiment_results

all_results = {}
for approach_name in APPROACHES.keys():
    exp_name = f"4cii_{approach_name}"
    try:
        r = load_experiment_results(exp_name, SEED, config_name=CONFIG_NAME, base_dir=RESULTS_BASE)
        if r is None or r.get("daily_returns") is None:
            print(f"  {approach_name}: not found, skipping")
            continue
        
        daily_returns = r["daily_returns"]
        metrics_raw = calc_metrics(daily_returns, approach_name)
        
        # Rebuild lightweight structure matching what run_approach() stores
        attn = r.get("attention_weights")
        avg_attn_sample = attn[:50].mean(axis=1) if attn is not None else None
        
        all_results[approach_name] = {
            "daily_returns": daily_returns,
            "metrics": metrics_raw,
            "history_loss": r.get("training_history", {}).get("loss", []),
            "history_val_loss": r.get("training_history", {}).get("val_loss", []),
            "concentration": {
                "avg_attn_sample": avg_attn_sample,
            },
            "kwargs": APPROACHES[approach_name],
        }
        print(f"  {approach_name}: loaded (Sharpe={metrics_raw['Sharpe']:.4f})")
    except Exception as e:
        print(f"  {approach_name}: error loading - {e}")

print(f"\nLoaded {len(all_results)} / {len(APPROACHES)} approaches")

In [ ]:
# Summary metrics table
metrics_list = [r["metrics"] for r in all_results.values()]
metrics_df = pd.DataFrame(metrics_list).set_index("Strategy")
for c in ["E[Ret.]","Vol.","Max DD","Hit Rate"]:
    if c in metrics_df.columns: metrics_df[c] = metrics_df[c].apply(lambda x: f"{x:.2%}")
for c in ["Sharpe","Sortino","Calmar","Avg P/L"]:
    if c in metrics_df.columns: metrics_df[c] = metrics_df[c].apply(lambda x: f"{x:.3f}")
display(metrics_df)

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, r in all_results.items():
    axes[0].plot(r["history_loss"], label=name, lw=1)
    axes[1].plot(r["history_val_loss"], label=name, lw=1)
axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.suptitle("4cii: Loss Curves", fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, r in all_results.items():
    axes[0].plot(r["history"].history["loss"], label=name, lw=1)
    axes[1].plot(r["history"].history["val_loss"], label=name, lw=1)
axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.suptitle("4cii: Loss Curves", fontweight="bold"); plt.tight_layout(); plt.show()

## 7. Attention Concentration Analysis

In [ ]:
# Compute concentration metrics WITHIN THE PEARSON MASK ONLY
# The mask zeros inflate Gini/zero_fraction for all approaches equally — misleading.
# We need to compare how concentrated attention is among the *actual neighbors*.

def compute_concentration_metrics(avg_attn_sample, adjacency_sample=None):
    """Compute Gini, effective neighbors, entropy WITHIN Pearson mask only.
    
    Args:
        avg_attn_sample: (samples, nodes, nodes) - head-averaged attention
        adjacency_sample: (samples, nodes, nodes) - Pearson mask (optional, for within-mask metrics)
    """
    all_metrics = {"gini": [], "eff_neighbors": [], "entropy": [], "max_weight": [],
                   "n_neighbors": [], "zero_within_mask": []}
    
    def gini(weights):
        sorted_w = np.sort(weights)
        n = len(sorted_w)
        if n == 0 or sorted_w.sum() == 0:
            return 0
        index = np.arange(1, n + 1)
        return (2 * np.sum(index * sorted_w) / (n * np.sum(sorted_w)) - (n + 1) / n)
    
    for s in range(avg_attn_sample.shape[0]):
        for i in range(avg_attn_sample.shape[1]):
            row = avg_attn_sample[s, i].copy()
            row[i] = 0  # exclude self
            
            # Get mask: which edges exist in Pearson graph?
            if adjacency_sample is not None:
                mask_row = adjacency_sample[s, i].copy()
                mask_row[i] = 0
                neighbors = mask_row > 0
            else:
                neighbors = row > 1e-10
            
            n_nbrs = neighbors.sum()
            if n_nbrs < 2:
                continue
            
            # Extract attention weights for ACTUAL neighbors only
            nbr_weights = row[neighbors]
            
            # Normalize to sum to 1 for fair comparison
            if nbr_weights.sum() > 0:
                nbr_weights_norm = nbr_weights / nbr_weights.sum()
            else:
                continue
            
            all_metrics["gini"].append(gini(nbr_weights_norm))
            all_metrics["n_neighbors"].append(n_nbrs)
            all_metrics["max_weight"].append(nbr_weights_norm.max())
            all_metrics["zero_within_mask"].append((nbr_weights < 1e-8).sum() / n_nbrs)
            
            # Entropy and effective neighbors (within mask)
            p = nbr_weights_norm[nbr_weights_norm > 1e-12]
            ent = -np.sum(p * np.log(p))
            all_metrics["entropy"].append(ent)
            all_metrics["eff_neighbors"].append(np.exp(ent))
    
    # Also compute: what fraction of MAX possible entropy is used?
    mean_n = np.mean(all_metrics["n_neighbors"])
    max_entropy = np.log(mean_n)  # entropy of uniform over mean_n neighbors
    mean_entropy = np.mean(all_metrics["entropy"])
    entropy_ratio = mean_entropy / max_entropy if max_entropy > 0 else 1.0
    
    return {
        "within_mask_gini": np.mean(all_metrics["gini"]),
        "within_mask_eff_neighbors": np.mean(all_metrics["eff_neighbors"]),
        "within_mask_entropy": mean_entropy,
        "within_mask_max_weight": np.mean(all_metrics["max_weight"]),
        "within_mask_zero_frac": np.mean(all_metrics["zero_within_mask"]),
        "mean_n_neighbors": mean_n,
        "entropy_ratio": entropy_ratio,  # 1.0 = perfectly uniform, 0.0 = all weight on one neighbor
    }

# Get adjacency samples matching the attention samples
adj_sample = test_data["adjacency"][:50]

concentration_results = {}
for name, r in all_results.items():
    avg_attn = r["concentration"]["avg_attn_sample"]
    metrics = compute_concentration_metrics(avg_attn, adj_sample[:len(avg_attn)])
    concentration_results[name] = metrics
    sharpe = r["metrics"]["Sharpe"]
    print(f"\n{name}:")
    print(f"  Within-mask Gini:         {metrics['within_mask_gini']:.4f}  (0=uniform, 1=one neighbor)")
    print(f"  Within-mask Eff.Neighbors: {metrics['within_mask_eff_neighbors']:.1f} / {metrics['mean_n_neighbors']:.1f} Pearson neighbors")
    print(f"  Entropy ratio:            {metrics['entropy_ratio']:.4f}  (1.0=uniform over all neighbors)")
    print(f"  Within-mask zero frac:    {metrics['within_mask_zero_frac']:.4f}  (fraction of Pearson edges zeroed)")
    print(f"  Max weight (within mask): {metrics['within_mask_max_weight']:.4f}")
    print(f"  Sharpe:                   {sharpe:.4f}")

In [ ]:
# Summary table: within-mask concentration metrics
conc_df = pd.DataFrame(concentration_results).T
conc_df.index.name = "Approach"
conc_df["sharpe"] = [all_results[n]["metrics"]["Sharpe"] for n in conc_df.index]

# Reorder columns for readability
col_order = ["entropy_ratio", "within_mask_gini", "within_mask_eff_neighbors",
             "mean_n_neighbors", "within_mask_max_weight", "within_mask_zero_frac", "sharpe"]
conc_df = conc_df[[c for c in col_order if c in conc_df.columns]]

display(conc_df.round(4))

print("\nKey column: entropy_ratio")
print("  1.0 = perfectly uniform attention over all Pearson neighbors")
print("  <0.9 = meaningful concentration (some neighbors weighted more)")
print("  <0.7 = strong concentration")
print("\nKey column: within_mask_zero_frac")
print("  >0 only for sparsemax/top-k (softmax can't produce exact zeros)")
print("  Shows fraction of Pearson-connected neighbors that get ZERO attention")

## 8. Visualisation

In [ ]:
# Concentration metrics bar chart (within-mask)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
approaches = list(concentration_results.keys())
colors = plt.cm.tab10(np.linspace(0, 1, len(approaches)))

for ax, metric, title in zip(axes,
    ["entropy_ratio", "within_mask_gini", "within_mask_eff_neighbors", "within_mask_max_weight"],
    ["Entropy Ratio (1=uniform)", "Gini (within mask)", "Eff. Neighbors (within mask)", "Max Weight (within mask)"]):
    vals = [concentration_results[a][metric] for a in approaches]
    bars = ax.bar(range(len(approaches)), vals, color=colors)
    ax.set_xticks(range(len(approaches)))
    ax.set_xticklabels(approaches, rotation=45, ha="right", fontsize=8)
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("4cii: Within-Mask Attention Concentration", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Sharpe vs entropy ratio scatter (within-mask)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
approaches = list(concentration_results.keys())
colors = plt.cm.tab10(np.linspace(0, 1, len(approaches)))

for ax, metric, xlabel in [
    (axes[0], "entropy_ratio", "Entropy Ratio (1=uniform, lower=more concentrated)"),
    (axes[1], "within_mask_gini", "Gini Coefficient within Mask"),
]:
    for i, name in enumerate(approaches):
        sharpe = all_results[name]["metrics"]["Sharpe"]
        val = concentration_results[name][metric]
        ax.scatter(val, sharpe, color=colors[i], s=100, zorder=5)
        ax.annotate(name, (val, sharpe), textcoords="offset points",
                    xytext=(8, 4), fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Sharpe Ratio")
    ax.grid(True, alpha=0.3)

plt.suptitle("4cii: Performance vs Attention Concentration (within Pearson mask)", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Attention heatmap grid: average attention for each approach
n_approaches = len(all_results)
fig, axes = plt.subplots(1, n_approaches, figsize=(4 * n_approaches, 4))
if n_approaches == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, all_results.items()):
    avg_attn = r["concentration"]["avg_attn_sample"].mean(axis=0)  # avg over samples
    sns.heatmap(avg_attn, ax=ax, cmap="viridis", vmin=0, cbar_kws={"shrink": 0.6})
    ax.set_title(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("4cii: Average Attention Heatmaps", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Attention weight distribution histograms
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, (name, r) in zip(axes, all_results.items()):
    avg_attn = r["concentration"]["avg_attn_sample"]
    weights = avg_attn[avg_attn > 1e-10].flatten()
    ax.hist(weights, bins=100, edgecolor="none", alpha=0.8)
    ax.set_title(f"{name}\n(non-zero weights: {len(weights):,})", fontsize=10)
    ax.set_xlabel("Attention Weight")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)

for ax in axes[len(all_results):]:
    ax.set_visible(False)

plt.suptitle("4cii: Attention Weight Distributions", fontweight="bold")
plt.tight_layout(); plt.show()